<a href="https://colab.research.google.com/github/monikanitcmca/GenAI/blob/main/Demo3_BiLSTM_CRF_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo 3: Named Entity Recognition with BiLSTM and BiLSTM-CRF

**What we build:** two neural NER models on the real CoNLL-2003 news dataset, then we judge them the right way.

1. **BiLSTM + softmax**: the network picks the best label for each word on its own.
2. **BiLSTM + CRF**: the same network, with a CRF on top that picks the best label sequence for the whole sentence.

**How we judge them:** entity-level precision, recall and F1 with the `seqeval` library. This is the evaluation from our CRF class: an entity counts as correct only if its type and its exact start and end are both right.

**The story to remember:** in the CRF class, a person wrote the clues (for example "starts with a capital letter"). Here the BiLSTM **learns its own clues**. The CRF part (transition table plus Viterbi) stays the same.

**Before you start:** in Colab, choose **Runtime > Change runtime type > T4 GPU**. The notebook also runs on CPU, only slower (about 1 to 2 minutes per training round instead of a few seconds).

| Part | What happens |
|---|---|
| 1 | Evaluation warm-up: word accuracy vs entity-level F1, by hand and with seqeval |
| 2 | Load CoNLL-2003 |
| 3 | Turn words and labels into numbers |
| 4 | Build the model (BiLSTM, with or without CRF) |
| 5 | Train with early stopping on dev F1 |
| 6 | Compare the two models on the test set |
| 7 | Try your own sentences |

## Part 0: Install and import

`pytorch-crf` gives us a ready, tested CRF layer. `seqeval` gives entity-level scores. PyTorch is already installed in Colab.

In [ ]:
!pip install -q pytorch-crf seqeval      # install the CRF layer and the NER scoring library

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import random                            # to shuffle training sentences
import time                              # to measure training time
import copy                              # to keep a copy of the best model
import urllib.request                    # to download the dataset files

import torch                             # PyTorch: tensors and neural networks
import torch.nn as nn                    # building blocks such as Embedding, LSTM, Linear
from torchcrf import CRF                 # the CRF layer from pytorch-crf
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report  # entity-level scores
from seqeval.metrics.sequence_labeling import get_entities                                   # lists the entities in a tag sequence

random.seed(42)                          # same shuffling every run
torch.manual_seed(42)                    # same starting weights every run

device = "cuda" if torch.cuda.is_available() else "cpu"   # use the GPU if Colab gives us one
print("Running on:", device)

Running on: cuda


## Part 1: Evaluation first

Before training anything, let us agree on how to judge a NER model. This is the same example we solved by hand in class.

Sentence: **"Priya Sharma met Rahul in New Delhi"**. The true entities are *Priya Sharma* (PER), *Rahul* (PER) and *New Delhi* (LOC).

In [ ]:
words = ["Priya", "Sharma", "met", "Rahul", "in", "New", "Delhi"]           # the sentence, one word per item
true_tags = ["B-PER", "I-PER", "O", "B-PER", "O", "B-LOC", "I-LOC"]          # the correct labels from a human

pred_1 = ["B-PER", "O",     "O", "B-PER", "O", "B-LOC", "I-LOC"]            # example 1: the model cut "Priya Sharma" short
pred_2 = ["B-PER", "I-PER", "B-PER", "B-PER", "O", "B-LOC", "O"]            # example 2: an extra entity and a broken one

print("True entities:  ", get_entities(true_tags))   # each entity as (type, start position, end position)
print("Example 1 found:", get_entities(pred_1))
print("Example 2 found:", get_entities(pred_2))

True entities:   [('PER', 0, 1), ('PER', 3, 3), ('LOC', 5, 6)]
Example 1 found: [('PER', 0, 0), ('PER', 3, 3), ('LOC', 5, 6)]
Example 2 found: [('PER', 0, 1), ('PER', 2, 2), ('PER', 3, 3), ('LOC', 5, 5)]


Read the output as `(type, first word position, last word position)`. Positions start at 0, so `('PER', 0, 1)` means words 0 to 1, "Priya Sharma".

Now compare **word accuracy** (share of words with the right label) with **entity-level** scores.

In [ ]:
def word_accuracy(true, pred):
    correct = sum(1 for t, p in zip(true, pred) if t == p)   # count words where the label matches
    return correct / len(true)                               # divide by the number of words

for name, pred in [("Example 1", pred_1), ("Example 2", pred_2)]:
    print(name)
    print("  word accuracy :", round(word_accuracy(true_tags, pred), 4))
    # seqeval expects a list of sentences, so we wrap each tag list in another list
    print("  precision     :", round(precision_score([true_tags], [pred]), 4))
    print("  recall        :", round(recall_score([true_tags], [pred]), 4))
    print("  F1            :", round(f1_score([true_tags], [pred]), 4))

Example 1
  word accuracy : 0.8571
  precision     : 0.6667
  recall        : 0.6667
  F1            : 0.6667
Example 2
  word accuracy : 0.7143
  precision     : 0.5
  recall        : 0.6667
  F1            : 0.5714


These match our hand calculation:

| | Word accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| Example 1 | 6 / 7 = 0.8571 | 2 / 3 = 0.6667 | 2 / 3 = 0.6667 | 0.6667 |
| Example 2 | 5 / 7 = 0.7143 | 2 / 4 = 0.5000 | 2 / 3 = 0.6667 | 2 × 0.5 × 0.6667 / (0.5 + 0.6667) = 0.5714 |

Word accuracy looks good (0.86) even though a whole name was cut. **From here on we only trust entity-level F1.**

## Part 2: Load CoNLL-2003

CoNLL-2003 is the standard English NER dataset: Reuters news sentences labelled with four types.

| Tag | Meaning | Example |
|---|---|---|
| PER | person | Peter Blackburn |
| ORG | organisation | European Commission |
| LOC | location | Brussels |
| MISC | other names | German, British |

Each line of the file has a word and three tags. We keep the word (first column) and the NER tag (last column). A blank line ends a sentence.

In [ ]:
BASE = "https://raw.githubusercontent.com/kyzhouhzau/BERT-NER/master/data/"   # a public copy of CoNLL-2003 in BIO format

def read_conll(file_name):
    text = urllib.request.urlopen(BASE + file_name).read().decode("utf-8")  # download the whole file as text
    sentences = []                                   # list of (words, tags) pairs
    words, tags = [], []                             # the sentence we are currently reading
    for line in text.split("\n"):                    # go line by line
        line = line.strip()                          # remove spaces and the line break at the ends
        if line == "" or line.startswith("-DOCSTART-"):   # blank line or document marker = end of a sentence
            if words:                                # only save if the sentence is not empty
                sentences.append((words, tags))
            words, tags = [], []                     # start a new sentence
        else:
            parts = line.split()                     # e.g. ["EU", "NNP", "B-NP", "B-ORG"]
            words.append(parts[0])                   # first column: the word
            tags.append(parts[-1])                   # last column: the NER tag
    if words:                                        # save the last sentence if the file did not end with a blank line
        sentences.append((words, tags))
    return sentences

train_data = read_conll("train.txt")   # used to learn the weights
dev_data = read_conll("dev.txt")       # used to pick the best training round (early stopping)
test_data = read_conll("test.txt")     # used once, at the very end, for the final score

print("Sentences  train:", len(train_data), " dev:", len(dev_data), " test:", len(test_data))
print("One example:", list(zip(*train_data[1])))   # show one sentence as (word, tag) pairs

Sentences  train: 14041  dev: 3250  test: 3453
One example: [('Peter', 'B-PER'), ('Blackburn', 'I-PER')]


## Part 3: Turn words and labels into numbers

Neural networks only understand numbers, so we build three lookup tables.

1. **word2id**: every training word that appears **at least twice** gets a number. We lowercase words so "Bank" and "bank" share one number. Two special entries: `<PAD>` = 0 fills short sentences, `<UNK>` = 1 stands for unknown words. Words seen only once in training also become `<UNK>`. This is on purpose: it lets the model practise on unknown words during training, so it learns that an unknown capitalised word is often a name (like "Ramesh" or "Koderma" later).
2. **case2id**: because we lowercased, we lose the capital-letter clue. So we add it back as a small extra feature with 5 values. This is the same "starts with a capital" clue from the CRF class, now given to the network.
3. **tag2id**: every label (B-PER, I-PER, O and so on) gets a number.

In [ ]:
from collections import Counter               # counts how often each item appears
word_counts = Counter(w.lower() for words, _ in train_data for w in words)   # how often each lowercased word appears in training

word2id = {"<PAD>": 0, "<UNK>": 1}            # start with the two special entries
for w, count in word_counts.items():
    if count >= 2:                             # keep words seen at least twice; the rest will become <UNK>
        word2id[w] = len(word2id)              # give it the next free number

def case_of(word):
    if word.islower():                         # "loan"
        return 1
    if word.istitle():                         # "Mumbai"
        return 2
    if word.isupper():                         # "SBI"
        return 3
    return 4                                   # numbers, punctuation, mixed like "iPhone" (0 is kept for padding)

tag_list = sorted({t for _, tags in train_data for t in tags})   # all different tags in the training data
tag2id = {t: i for i, t in enumerate(tag_list)}                   # tag -> number
id2tag = {i: t for t, i in tag2id.items()}                        # number -> tag, to read predictions back

print("Vocabulary size:", len(word2id))
print("Tags:", tag_list)
print("Case of 'loan', 'Mumbai', 'SBI', '2024':", case_of("loan"), case_of("Mumbai"), case_of("SBI"), case_of("2024"))

Vocabulary size: 10951
Tags: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Case of 'loan', 'Mumbai', 'SBI', '2024': 1 2 3 4


Sentences have different lengths, but a batch must be a rectangle. So we **pad** short sentences with 0 and keep a **mask** that says which positions are real words (True) and which are padding (False).

In [ ]:
def make_batch(batch):
    lengths = [len(words) for words, _ in batch]          # length of each sentence in the batch
    max_len = max(lengths)                                # the longest one sets the width
    word_ids = torch.zeros(len(batch), max_len, dtype=torch.long)   # filled with 0 = <PAD>
    case_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    tag_ids = torch.zeros(len(batch), max_len, dtype=torch.long)    # padding tag 0 is ignored by the mask
    for i, (words, tags) in enumerate(batch):
        for j, (w, t) in enumerate(zip(words, tags)):
            word_ids[i, j] = word2id.get(w.lower(), 1)    # 1 = <UNK> if the word was not in training
            case_ids[i, j] = case_of(w)
            tag_ids[i, j] = tag2id.get(t, tag2id["O"])    # unknown tags (should not happen) become O
    mask = word_ids != 0                                  # True for real words, False for padding
    mask[:, 0] = True                                     # the CRF needs the first position to be real in every row
    return word_ids.to(device), case_ids.to(device), tag_ids.to(device), mask.to(device), torch.tensor(lengths)

w, c, t, m, L = make_batch(train_data[:2])   # try it on two sentences
print("word ids:\n", w)
print("mask:\n", m)

word ids:
 tensor([[ 2,  1,  3,  4,  5,  6,  7,  8,  9],
        [10, 11,  0,  0,  0,  0,  0,  0,  0]], device='cuda:0')
mask:
 tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True, False, False, False, False, False, False, False]],
       device='cuda:0')


## Part 4: The model

One class covers both models. The switch `use_crf` decides the last step.

```
words + case  ->  embeddings  ->  BiLSTM (reads left-to-right and right-to-left)  ->  Linear  ->  one score per label per word
                                                                                                   |
                                         use_crf = False: take the highest score at each word (softmax, word by word)
                                         use_crf = True : CRF adds the transition table, Viterbi picks the best whole sequence
```

The scores from the Linear layer are the **emission scores**, the network's version of the "clue points" from the CRF class.

In [ ]:
class NERTagger(nn.Module):
    def __init__(self, vocab_size, num_tags, use_crf):
        super().__init__()
        self.use_crf = use_crf                                       # remember which version we are
        self.word_emb = nn.Embedding(vocab_size, 100, padding_idx=0) # each word -> 100 numbers (learnt)
        self.case_emb = nn.Embedding(5, 10, padding_idx=0)           # each case type -> 10 numbers (learnt)
        self.dropout = nn.Dropout(0.5)                               # randomly hides half the values while training, to avoid overfitting
        self.lstm = nn.LSTM(110, 100, batch_first=True, bidirectional=True)  # 100 + 10 inputs, 100 memory units each direction
        self.to_tags = nn.Linear(200, num_tags)                      # 100 forward + 100 backward -> one score per tag
        if use_crf:
            self.crf = CRF(num_tags, batch_first=True)               # learns the tag-to-tag transition table

    def emissions(self, word_ids, case_ids, lengths):
        x = torch.cat([self.word_emb(word_ids), self.case_emb(case_ids)], dim=-1)  # join word and case vectors: 110 numbers per word
        x = self.dropout(x)
        # "packing" tells the LSTM the real length of each sentence, so the backward reader starts at the last real word, not at padding
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)                                   # run the BiLSTM
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=word_ids.shape[1])  # back to a rectangle
        out = self.dropout(out)
        return self.to_tags(out)                                     # shape: sentences x words x tags

    def loss(self, word_ids, case_ids, tag_ids, mask, lengths):
        scores = self.emissions(word_ids, case_ids, lengths)
        if self.use_crf:
            return -self.crf(scores, tag_ids, mask=mask, reduction="mean")   # CRF gives log-likelihood; minus makes it a loss
        loss_fn = nn.CrossEntropyLoss()                              # plain softmax loss, word by word
        return loss_fn(scores[mask], tag_ids[mask])                  # only real words count, padding is skipped

    def predict(self, word_ids, case_ids, mask, lengths):
        scores = self.emissions(word_ids, case_ids, lengths)
        if self.use_crf:
            return self.crf.decode(scores, mask=mask)                # Viterbi: best whole sequence per sentence
        best = scores.argmax(dim=-1)                                 # best tag at each word, chosen alone
        return [best[i, :n].tolist() for i, n in enumerate(lengths.tolist())]   # cut off the padding

model = NERTagger(len(word2id), len(tag2id), use_crf=True)
print(model)
print("Trainable numbers in the model:", sum(p.numel() for p in model.parameters()))

NERTagger(
  (word_emb): Embedding(10951, 100, padding_idx=0)
  (case_emb): Embedding(5, 10, padding_idx=0)
  (dropout): Dropout(p=0.5, inplace=False)
  (lstm): LSTM(110, 100, batch_first=True, bidirectional=True)
  (to_tags): Linear(in_features=200, out_features=9, bias=True)
  (crf): CRF(num_tags=9)
)
Trainable numbers in the model: 1266658


## Part 5: Training with early stopping on dev F1

The recipe for each round (epoch):

1. Shuffle the training sentences and cut them into batches of 32.
2. For each batch: compute the loss, backpropagate, **clip gradients at 5** (LSTMs sometimes produce huge gradients), update the weights.
3. At the end of the round, compute **entity-level F1 on the dev set**. If it is the best so far, keep a copy of the model.

We keep the best dev model, not the last one, because training loss keeps falling after real accuracy stops improving.

In [ ]:
def predict_tags(model, data, batch_size=128):
    model.eval()                                          # switch dropout off for prediction
    all_preds = []
    with torch.no_grad():                                 # no gradients needed: faster, less memory
        for i in range(0, len(data), batch_size):
            batch = data[i:i + batch_size]
            w, c, t, m, L = make_batch(batch)
            for ids in model.predict(w, c, m, L):         # one list of tag numbers per sentence
                all_preds.append([id2tag[k] for k in ids])   # numbers -> tag names
    return all_preds

def train_model(use_crf, epochs=10, batch_size=32):
    torch.manual_seed(42)                                 # same start for both models, for a fair comparison
    model = NERTagger(len(word2id), len(tag2id), use_crf).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)   # Adam with the usual learning rate
    dev_true = [tags for _, tags in dev_data]             # correct dev labels, to score against
    best_f1, best_state = -1, None
    for epoch in range(1, epochs + 1):
        start = time.time()
        model.train()                                     # switch dropout on for training
        data = train_data[:]                              # copy the list so shuffling does not change the original
        random.shuffle(data)
        total_loss = 0
        for i in range(0, len(data), batch_size):
            w, c, t, m, L = make_batch(data[i:i + batch_size])
            loss = model.loss(w, c, t, m, L)              # how wrong are we on this batch?
            optimizer.zero_grad()                         # clear old gradients
            loss.backward()                               # compute new gradients
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)   # cap very large gradients
            optimizer.step()                              # update the weights
            total_loss += loss.item()
        dev_f1 = f1_score(dev_true, predict_tags(model, dev_data))   # entity-level F1 on dev
        marker = ""
        if dev_f1 > best_f1:                              # new best? keep a copy
            best_f1, best_state = dev_f1, copy.deepcopy(model.state_dict())
            marker = "  <- best so far"
        print(f"epoch {epoch:2d}  loss {total_loss:9.1f}  dev F1 {dev_f1:.4f}  ({time.time() - start:.0f} s){marker}")
    model.load_state_dict(best_state)                     # go back to the best round
    return model

### Model 1: BiLSTM + softmax (no CRF)

Each word's label is chosen on its own. Nothing stops the model from writing `O I-ORG`, which is an impossible sequence.

In [ ]:
softmax_model = train_model(use_crf=False, epochs=10)

epoch  1  loss     220.3  dev F1 0.5394  (11 s)  <- best so far
epoch  2  loss     121.3  dev F1 0.6676  (10 s)  <- best so far
epoch  3  loss      96.2  dev F1 0.7196  (10 s)  <- best so far
epoch  4  loss      81.3  dev F1 0.7544  (9 s)  <- best so far
epoch  5  loss      70.8  dev F1 0.7679  (10 s)  <- best so far
epoch  6  loss      63.1  dev F1 0.7875  (10 s)  <- best so far
epoch  7  loss      57.2  dev F1 0.8037  (10 s)  <- best so far
epoch  8  loss      52.1  dev F1 0.8198  (10 s)  <- best so far
epoch  9  loss      48.3  dev F1 0.8260  (9 s)  <- best so far
epoch 10  loss      44.6  dev F1 0.8321  (10 s)  <- best so far


### Model 2: BiLSTM + CRF

Same network, same data, same settings. The only change is the CRF on top.

In [ ]:
crf_model = train_model(use_crf=True, epochs=10)

epoch  1  loss    2855.9  dev F1 0.6195  (29 s)  <- best so far
epoch  2  loss    1488.2  dev F1 0.7042  (29 s)  <- best so far
epoch  3  loss    1141.4  dev F1 0.7480  (29 s)  <- best so far
epoch  4  loss     950.7  dev F1 0.7776  (29 s)  <- best so far
epoch  5  loss     812.2  dev F1 0.8018  (29 s)  <- best so far
epoch  6  loss     718.0  dev F1 0.8129  (29 s)  <- best so far
epoch  7  loss     639.8  dev F1 0.8263  (30 s)  <- best so far
epoch  8  loss     573.3  dev F1 0.8342  (29 s)  <- best so far
epoch  9  loss     530.7  dev F1 0.8463  (29 s)  <- best so far
epoch 10  loss     482.6  dev F1 0.8546  (30 s)  <- best so far


## Part 6: Compare on the test set

We use the test set only now, once, for the final score. We report:

* **Entity-level precision, recall and F1** (seqeval), overall and per type.
* **Illegal tags**: how many times an `I-` tag appears without a `B-` or `I-` of the same type just before it. Example: `O I-ORG`.

In [ ]:
def count_illegal(tag_sequences):
    bad = 0
    for tags in tag_sequences:
        prev = "O"                                        # the tag before the first word counts as O
        for t in tags:
            if t.startswith("I-") and prev[2:] != t[2:]:  # I-ORG must follow B-ORG or I-ORG
                bad += 1
            prev = t
    return bad

test_true = [tags for _, tags in test_data]               # correct test labels

for name, m in [("BiLSTM + softmax", softmax_model), ("BiLSTM + CRF", crf_model)]:
    preds = predict_tags(m, test_data)
    print("=" * 60)
    print(name)
    print(f"precision {precision_score(test_true, preds):.4f}   recall {recall_score(test_true, preds):.4f}   F1 {f1_score(test_true, preds):.4f}")
    print("illegal tag sequences:", count_illegal(preds), "   (in the true labels:", count_illegal(test_true), ")")
    print(classification_report(test_true, preds, digits=4))

BiLSTM + softmax
precision 0.7551   recall 0.7773   F1 0.7660
illegal tag sequences: 220    (in the true labels: 0 )
              precision    recall  f1-score   support

         LOC     0.8216    0.8034    0.8124      1668
        MISC     0.6884    0.6610    0.6744       702
         ORG     0.7077    0.7098    0.7087      1661
         PER     0.7634    0.8701    0.8133      1617

   micro avg     0.7551    0.7773    0.7660      5648
   macro avg     0.7453    0.7611    0.7522      5648
weighted avg     0.7549    0.7773    0.7650      5648

BiLSTM + CRF
precision 0.7844   recall 0.7891   F1 0.7868
illegal tag sequences: 0    (in the true labels: 0 )
              precision    recall  f1-score   support

         LOC     0.8507    0.8333    0.8419      1668
        MISC     0.7194    0.6538    0.6851       702
         ORG     0.7078    0.7556    0.7309      1661
         PER     0.8265    0.8367    0.8316      1617

   micro avg     0.7844    0.7891    0.7868      5648
   macro av

**How to read this.** In our own test run we got: BiLSTM + softmax F1 **0.775** with **220** illegal sequences, and BiLSTM + CRF F1 **0.793** with **1** illegal sequence. Your numbers will differ a little from run to run.

* The CRF model should have **almost no illegal sequences**, because its transition table learnt that `O -> I-ORG` never happens. The softmax model has many.
* The CRF model usually has a **higher F1**, mostly because it fixes broken multi-word names such as "European Commission".
* **LOC and PER** usually score highest. **MISC** is hardest because it is a mixed bag (nationalities, events, products).
* Test F1 is lower than dev F1 (on CoNLL-2003 the test set is known to be harder). Our scores are well below the published BiLSTM-CRF result of about 90. The papers added **pretrained word embeddings** (such as GloVe) and **character-level features**. We kept the model small and simple on purpose. A fine-tuned BERT tagger usually reaches about 91 to 92.

Note: seqeval's default mode is lenient. It reads a stray `I-ORG` after `O` as the start of a new entity. That is why the softmax model is not punished even more.

### Look inside: the learnt transition table

The CRF learnt one score for every "from tag -> to tag" pair. High score = common, very negative = almost never. This is the same table we filled in by hand in class, now learnt from data.

In [ ]:
trans = crf_model.crf.transitions.detach().cpu()          # a tags x tags table of learnt scores
for a, b in [("O", "B-ORG"), ("B-ORG", "I-ORG"), ("I-ORG", "I-ORG"), ("O", "I-ORG"), ("B-PER", "I-ORG")]:
    print(f"{a:6s} -> {b:6s}: {trans[tag2id[a], tag2id[b]].item():6.2f}")

O      -> B-ORG :   0.36
B-ORG  -> I-ORG :   1.67
I-ORG  -> I-ORG :   1.21
O      -> I-ORG :  -2.05
B-PER  -> I-ORG :  -2.25


You should see positive scores for `B-ORG -> I-ORG` and `I-ORG -> I-ORG`, and clearly negative scores for `O -> I-ORG` and `B-PER -> I-ORG`. Nobody wrote these rules; the CRF learnt them from the labelled data.

## Part 7: Try your own sentences

Split the sentence into words with spaces (put a space before the full stop too). Words never seen in training become `<UNK>`, so the model must rely on the case feature and the words around them.

In [ ]:
def tag_sentence(model, sentence):
    words = sentence.split()                              # simple split on spaces
    tags = predict_tags(model, [(words, ["O"] * len(words))])[0]   # dummy labels, only the words are used
    return list(zip(words, tags))

examples = [
    "Ramesh Kumar took a loan from State Bank of India in Mumbai .",
    "Priya Sharma met Rahul in New Delhi .",
    "Infosys opened a new office in Pune last week .",
]
for s in examples:
    print("softmax:", tag_sentence(softmax_model, s))
    print("CRF    :", tag_sentence(crf_model, s))
    print()

softmax: [('Ramesh', 'B-PER'), ('Kumar', 'I-PER'), ('took', 'O'), ('a', 'O'), ('loan', 'O'), ('from', 'O'), ('State', 'B-ORG'), ('Bank', 'I-ORG'), ('of', 'I-ORG'), ('India', 'I-ORG'), ('in', 'O'), ('Mumbai', 'B-LOC'), ('.', 'O')]
CRF    : [('Ramesh', 'B-PER'), ('Kumar', 'I-PER'), ('took', 'O'), ('a', 'O'), ('loan', 'O'), ('from', 'O'), ('State', 'B-ORG'), ('Bank', 'I-ORG'), ('of', 'I-ORG'), ('India', 'I-ORG'), ('in', 'O'), ('Mumbai', 'B-LOC'), ('.', 'O')]

softmax: [('Priya', 'B-PER'), ('Sharma', 'I-PER'), ('met', 'O'), ('Rahul', 'B-LOC'), ('in', 'O'), ('New', 'B-LOC'), ('Delhi', 'I-LOC'), ('.', 'O')]
CRF    : [('Priya', 'B-PER'), ('Sharma', 'I-PER'), ('met', 'O'), ('Rahul', 'B-LOC'), ('in', 'O'), ('New', 'B-LOC'), ('Delhi', 'I-LOC'), ('.', 'O')]

softmax: [('Infosys', 'O'), ('opened', 'O'), ('a', 'O'), ('new', 'O'), ('office', 'O'), ('in', 'O'), ('Pune', 'B-LOC'), ('last', 'O'), ('week', 'O'), ('.', 'O')]
CRF    : [('Infosys', 'B-ORG'), ('opened', 'O'), ('a', 'O'), ('new', 'O'), ('off

Things to discuss with learners:

* The data is **news from the 1990s**, mostly European and American names. Indian names such as "Ramesh Kumar" or places such as "Pune" may be unknown words. The model can still guess from the capital letters and the words around them, but it will make mistakes.
* This is why domain data matters. For Indian banking text you would label your own sentences, or start from a multilingual pretrained model.

## Summary

| | BiLSTM + softmax | BiLSTM + CRF |
|---|---|---|
| Who writes the clues | the network | the network |
| How labels are chosen | each word alone | whole sentence (Viterbi) |
| Illegal sequences | many | almost none |
| Entity-level F1 | lower | higher |

**Today:** a fine-tuned BERT tagger is the usual production choice, and LLMs can extract entities as JSON with no training. BiLSTM-CRF still fits when you need a small model that runs fast on a CPU. The evaluation you used here, entity-level F1 with seqeval, is the same for all of them.